# Switch to a local model (Ollama)

Ollama runs open-weight models in the codespace behind an OpenAI-compatible API: we change only `base_url` and `model`, then evaluate again.

In this notebook, we will:
1. Check that Ollama runs.
2. Point the OpenAI client to `localhost:11434/v1`.
3. Rerun the review classification on 20 reviews.
4. Compare accuracy, speed and cost with `gpt-6-luna`.

**Time:** about 30 minutes, self-study. Tip: a 4-core codespace is faster.

## Libraries and settings

Both models receive the same locally anonymized review text and an explicit final-label prompt. The OpenAI helper is called with `use_cache=False` to measure request latency: **rerunning the API timing cell incurs charges**. Local inference has no API fee, but hardware, electricity and Codespaces runtime cost money. This small sample measures agreement with API-generated silver labels, not independent human accuracy.

We intentionally use `qwen3:1.7b` (about 1.4 GB) for small CPU Codespaces. The lecture's `qwen3.5:2b` is a larger alternative, not the model measured here.


In [ ]:
# Libraries
from llm_course import DATA, ROOT, ask, parse_label
import os
import re
import time
import platform
import warnings
from datetime import datetime
from platform import python_version

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import accuracy_score


# Settings
MODEL = 'gpt-6-luna'
PRICE_OUTPUT_PER_M = 0.50       # USD per 1M output tokens, including reasoning
PRICE_INPUT_PER_M = 0.10        # USD per 1M input tokens
LOCAL_MODEL = 'qwen3:1.7b'

# Key from a local .env file (only needed outside Codespaces)
load_dotenv(ROOT / '.env')
print('API key found:', bool(os.environ.get('OPENAI_API_KEY')))

# Show current working directory
print(os.getcwd())

## Step 1: Check that Ollama runs

Installs Ollama if missing (about 2 minutes), starts the server and downloads the model (about 1.4 GB, once).

In [ ]:
!which zstd > /dev/null || (sudo apt-get update -qq && sudo apt-get install -y -qq zstd)
!which ollama || curl -fsSL https://ollama.com/install.sh | sh
!pgrep -x ollama > /dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)
!sleep 3 && ollama pull {LOCAL_MODEL} && ollama list

## Step 2: Point the OpenAI client to localhost:11434/v1

Ollama ignores the key, but the client needs one.

In [ ]:
local = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

r = local.chat.completions.create(
    model=LOCAL_MODEL,
    messages=[{'role': 'user', 'content': 'Name one typical dish in the IKEA restaurant. One sentence.'}],
)
print(r.choices[0].message.content)

## Step 3: Rerun the review classification on 20 reviews

20 reviews with silver labels from `03_Classify_Reviews`, the same one-word prompt for both models.

In [ ]:
reviews_path = DATA / 'processed/ikea_reviews_anonymized.csv'
if not reviews_path.exists():
    raise FileNotFoundError('Run the anonymization and save cells in 01_Explore_Yelp_Data first.')

df = pd.read_csv(reviews_path, encoding='utf-8')
silver_path = DATA / 'processed/silver_labels.csv'
if not silver_path.exists():
    raise FileNotFoundError('Run 03_Classify_Reviews first to create silver_labels.csv.')
silver = pd.read_csv(silver_path)
matched = silver[silver['silver'].isin(['positive', 'negative', 'mixed'])].merge(df, on='review_id', validate='one_to_one')
if len(matched) < 20:
    raise ValueError('At least 20 valid silver labels are required.')
test = matched.sample(20, random_state=42).copy()
PROMPT = 'Classify sentiment as positive, negative or mixed. Answer only: Overall: <label>.\n'


## Use the texts prepared in notebook 01

Load the locally anonymized copies saved by **01_Explore_Yelp_Data**. Inspect and correct remaining identifiers there before saving; rerun the load cells here after making corrections. These are private working files with original IDs/metadata retained for local joins, not a dataset to publish. The API helpers repeat local redaction immediately before sending prompt text, examples and history. If the prepared files are missing, return to notebook 01 and run its preparation and save cells.


In [ ]:
test[['stars', 'text']].head(10)

In [ ]:
# Local model: Chat Completions API
labels, start = [], time.perf_counter()
for t in test['text']:
    r = local.chat.completions.create(model=LOCAL_MODEL, messages=[{'role': 'user', 'content': PROMPT + t}])
    labels.append(parse_label(r.choices[0].message.content))
test['local'] = labels
local_seconds = (time.perf_counter() - start) / len(test)

In [ ]:
# Uncached requests: every rerun incurs API charges. Privacy preparation is cached in memory.
labels, input_tokens, output_tokens, cached_tokens, cache_write_tokens = [], 0, 0, 0, 0
start = time.perf_counter()
for t in test['text']:
    r = ask(model=MODEL, input=PROMPT + t, use_cache=False)
    labels.append(parse_label(r.output_text) if r.status == 'completed' else 'incomplete')
    input_tokens += r.usage.input_tokens
    output_tokens += r.usage.output_tokens  # Already includes reasoning; do not add it again.
    details = r.usage.input_tokens_details
    cached_tokens += getattr(details, 'cached_tokens', 0) or 0
    cache_write_tokens += getattr(details, 'cache_write_tokens', 0) or 0
test['api'] = labels
api_seconds = (time.perf_counter() - start) / len(test)
api_cost = ((input_tokens - cached_tokens - cache_write_tokens) * PRICE_INPUT_PER_M + cached_tokens * 0.01 + cache_write_tokens * 0.125 + output_tokens * PRICE_OUTPUT_PER_M) / 1e6


## Step 4: Compare accuracy, speed and cost

In [ ]:
comparison = pd.DataFrame(
    {
        LOCAL_MODEL: [accuracy_score(test['silver'], test['local']), local_seconds, 0.0],
        MODEL: [accuracy_score(test['silver'], test['api']), api_seconds, api_cost],
    },
    index=['agreement with silver', 'seconds per review', 'API cost (USD)'],
).round(4)
print('Parsing/status failures:', {col: int((~test[col].isin(['positive','negative','mixed'])).sum()) for col in ['local', 'api']})
print('Input, cached input, cache writes, output tokens:', input_tokens, cached_tokens, cache_write_tokens, output_tokens)

comparison


## Interpret the comparison

Twenty reviews are too few for a confident provider decision, and the reference favours the model that generated it. Compare both models against independently checked human labels before making a claim about accuracy. Keep invalid outputs in the denominator. Latencies include client/request overhead and can vary with hardware, load, thinking mode and retries; throughput under concurrency is a separate experiment. Repeat measurements only when needed, since uncached API calls cost money.

For one million reviews per month, first test a simple baseline (keyword rules or TF-IDF with a supervised classifier), then compare quality, throughput, total cost, privacy and maintenance. Prefer local inference if sufficiently de-identifying the data is not feasible. API pricing here is a dated Standard-rate estimate (10 Nov 2026); verify provider pricing before budgeting. Local hardware and operations are excluded from API fees.


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')